In [ ]:
using MPSDynamics, ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b_0 = 7.0515*2*pi

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.1123
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 36.924*2*pi


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.3
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
α_car = 3.413
ϵ_d=2*α_car*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

# Check chaos for Mean-field

In [ ]:
ϵ_0_list = vcat(0.5:.1:1,1:.1:0.6pi)
κ_b_list = 0.05:0.01:0.2

In [ ]:
Lyap_coef_chaos = Array{Float64}(undef, length(ϵ_0_list), length(κ_b_list))
drive = DriveTwoModes(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
drive_prepared = prepare(drive)
for i in ProgressBar(1:length(κ_b_list))
    κ_b = κ_b_list[i]
    t_max = 10/κ_b
    t_w = 30/κ_b
    ϕ_a_0, q_a_0=0.0, 0.0
    ϕ_b_0, q_b_0=0.0, 0.0

    system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
    environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

    system_prepared = prepare(system)
    environment_prepared = prepare(system,environment)

    dt = get_dt(system, environment)
    u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
    u_0_new = vcat(u_0,u_dot_0)
    for j in 1:length(ϵ_0_list)
        ϵ_0_j = ϵ_0_list[j]
        drive.ϵ_0 = ϵ_0_j
        g2_j = besselj(1,ϵ_0_j)*E_J*φ_a^2*φ_b # (Leon's paper)
        ϵ_d_j = 2*α_car*g2_j
        drive.ϵ_d = ϵ_d_j
        drive_prepared.drive = drive
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        problem = CoupledODEs(equation_1ODE!, u_0_new, p)

        Lyap_coef_chaos[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
    end
end

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b
g2_lis = 0:maximum(g2_list)/100:maximum(g2_list)
heatmap(g2_list/2/pi*1e3,κ_b_list,Lyap_coef_chaos)
plot!(g2_lis/2pi*1e3, g2_lis*sqrt(α_car)/4,label="")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\kappa_b",
    size=(size_px_car[1],size_px_car[2]),
    colorbar_title="\nLyapunov exp.", title = "Mean field ",
    xlims=(minimum(g2_list/2pi*1e3),maximum(g2_list/2pi*1e3)),ylims=(minimum(κ_b_list),maximum(κ_b_list)),
    clims=(minimum(Lyap_coef_chaos),maximum(Lyap_coef_chaos)),
    right_margin=10mm)
#savefig("../../figures/Two-photon/heatmap_lyapunov_coef.pdf")

In [ ]:
Lyap_coef_chaos_sign = map(x -> x>0 ? 1 : -1, Lyap_coef_chaos);

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b
g2_lis = 0:maximum(g2_list)/100:maximum(g2_list)
heatmap(g2_list/2/pi*1e3,κ_b_list,Lyap_coef_chaos_sign)
plot!(g2_lis/2pi*1e3, g2_lis*sqrt(α_car)/4,label="")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\kappa_b",
    size=(size_px_car[1],size_px_car[2]),
    colorbar_title="\nLyapunov exp.", title = "Mean field ",
    xlims=(minimum(g2_list/2pi*1e3),maximum(g2_list/2pi*1e3)),ylims=(minimum(κ_b_list),maximum(κ_b_list)),
    clims=(minimum(Lyap_coef_chaos_sign),maximum(Lyap_coef_chaos_sign)),
    right_margin=10mm)
#savefig("../../figures/Two-photon/heatmap_lyapunov_coef.pdf")